# LeetCode 822 — Card Flipping Game bằng Python

Link đề: https://leetcode.com/problems/card-flipping-game/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        ...
```

Ghi chú: bài này nhìn như bài "thử mọi cách lật" (`2^n` khả năng) nhưng thực ra có một **quan sát then chốt** biến nó thành bài lọc tập hợp chỉ vài dòng: *một số không thể "tốt" khi và chỉ khi có một lá bài mà mặt trước và mặt sau đều là số đó.* Notebook đi từ vét cạn `2^n` (làm oracle), qua cách "thử từng số ứng viên và dựng cách lật", rồi tới cách dùng tập hợp, sắp xếp, và mảng đếm.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` lá bài, mỗi lá có số ở mặt trước và mặt sau; ta được lật tuỳ ý; một số là **tốt** nếu nó nằm ở mặt **dưới** của ít nhất một lá và **không** nằm ở mặt **trên** của lá nào. Tìm số tốt nhỏ nhất có thể đạt được (hoặc `0`).
2. Nắm công cụ Python cần dùng: `zip`, set comprehension, phép hợp/hiệu tập hợp, `itertools.chain`, `min(..., default=0)`, duyệt mọi tập con bằng mặt nạ bit, mảng đánh dấu theo giá trị.
3. Đi từ **vét cạn `2^n` cách lật** → **thử từng số ứng viên rồi dựng cách lật** → **tập số bị cấm bằng hash set** → **sắp xếp rồi lấy số hợp lệ đầu tiên** → **mảng đánh dấu theo giá trị**.
4. Hiểu mấu chốt: số `x` **bị cấm** ⇔ tồn tại lá bài có `fronts[i] == backs[i] == x`; mọi số xuất hiện trên bài mà không bị cấm đều đạt được là số tốt.
5. Tránh các lỗi kinh điển: quên mặt sau khi tìm số bị cấm, cấm nhầm số xuất hiện trên nhiều lá, chỉ xét một mặt khi tìm ứng viên, `min` của dãy rỗng gây lỗi, và không loại số bị cấm khỏi ứng viên.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `fronts` và `backs` độ dài `n` (đánh chỉ số từ `0`). Lá bài thứ `i` có số `fronts[i]` in ở mặt trước và `backs[i]` ở mặt sau. Ban đầu mặt trước hướng **lên**, mặt sau hướng **xuống**. Bạn được lật **bao nhiêu lá tuỳ ý** (kể cả không lật lá nào).

Sau khi lật, một số nguyên được gọi là **tốt** nếu nó nằm ở mặt **dưới** của **ít nhất một** lá bài và **không** nằm ở mặt **trên** của **bất kỳ** lá bài nào.

Hãy trả về **số tốt nhỏ nhất** có thể có sau khi lật. Nếu không thể có số tốt nào, trả về `0`.

**Ví dụ 1**

- Input: `fronts = [1,2,4,4,7]`, `backs = [1,3,4,1,3]`
- Output: `2`
- Giải thích: lật lá thứ hai thì mặt trên là `[1,3,4,4,7]` và mặt dưới là `[1,2,4,1,3]`. Số `2` nằm ở mặt dưới mà không nằm ở mặt trên nên là số tốt. Không thể có số tốt nào nhỏ hơn.

**Ví dụ 2**

- Input: `fronts = [1]`, `backs = [1]`
- Output: `0`
- Giải thích: dù lật hay không, số `1` luôn nằm cả trên lẫn dưới; không có số tốt nào.

**Ràng buộc:**

- `n == fronts.length == backs.length`
- `1 <= n <= 1000`
- `1 <= fronts[i], backs[i] <= 2000`

In [1]:
from typing import List
from itertools import chain
import random

examples = [
    ([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2),
    ([1], [1], 0),
]
for fronts, backs, expected in examples:
    print("fronts =", fronts, "| backs =", backs, "| kỳ vọng:", expected)

fronts = [1, 2, 4, 4, 7] | backs = [1, 3, 4, 1, 3] | kỳ vọng: 2
fronts = [1] | backs = [1] | kỳ vọng: 0


## 3. Kiến thức nền

### 3.1. `zip`: duyệt hai mảng song song

`zip(a, b)` ghép phần tử cùng vị trí của hai dãy thành từng cặp — đúng thứ ta cần để nhìn **cả hai mặt** của từng lá bài.

In [2]:
fronts, backs = examples[0][0], examples[0][1]
for i, (a, b) in enumerate(zip(fronts, backs)):
    print(f"lá {i}: trước = {a}, sau = {b}", "<- hai mặt GIỐNG nhau" if a == b else "")

lá 0: trước = 1, sau = 1 <- hai mặt GIỐNG nhau
lá 1: trước = 2, sau = 3 
lá 2: trước = 4, sau = 4 <- hai mặt GIỐNG nhau
lá 3: trước = 4, sau = 1 
lá 4: trước = 7, sau = 3 


### 3.2. Tập hợp (set): thành viên và phép toán

`set` lưu các phần tử **không trùng**, kiểm tra `x in s` trung bình `O(1)`. Set comprehension `{a for ... if ...}` dựng tập theo điều kiện. Phép **hợp** `A | B` (có ở A hoặc B), phép **hiệu** `A - B` (có ở A mà không có ở B).

In [3]:
A = {1, 2, 4, 7}
B = {1, 3, 4}
print("A | B =", sorted(A | B), "| A - B =", sorted(A - B), "| 2 in A:", 2 in A, "| 3 in A:", 3 in A)
same = {a for a, b in zip(fronts, backs) if a == b}
print("các số xuất hiện ở lá có hai mặt giống nhau:", sorted(same))

A | B = [1, 2, 3, 4, 7] | A - B = [2, 7] | 2 in A: True | 3 in A: False
các số xuất hiện ở lá có hai mặt giống nhau: [1, 4]


### 3.3. `itertools.chain`, `min(..., default=0)`

`chain(a, b)` nối hai dãy thành một mà không sao chép. `min` của một dãy **rỗng** báo `ValueError`; tham số `default=0` trả `0` khi dãy rỗng — rất hợp với đề ("không có số tốt thì trả 0").

In [4]:
print(list(chain([1, 2], [3, 4])))
print(min([5, 3, 9], default=0))
print(min([], default=0))
try:
    min([])
except ValueError as e:
    print("min([]) ->", "ValueError:", e)

[1, 2, 3, 4]
3
0
min([]) -> ValueError: min() iterable argument is empty


### 3.4. Mặt nạ bit để liệt kê mọi cách lật

Có `n` lá, mỗi lá "lật" hoặc "không lật" → `2^n` cách. Mỗi cách mã hoá bằng một số nguyên `mask` từ `0` đến `2^n - 1`: bit thứ `i` bằng `1` nghĩa là lật lá `i`. `mask >> i & 1` đọc bit `i`.

In [5]:
n = 3
for mask in range(1 << n):
    flips = [(mask >> i) & 1 for i in range(n)]
    print(f"mask = {mask} -> nhị phân {mask:03b} -> lật các lá {[i for i in range(n) if flips[i]]}")

mask = 0 -> nhị phân 000 -> lật các lá []
mask = 1 -> nhị phân 001 -> lật các lá [0]
mask = 2 -> nhị phân 010 -> lật các lá [1]
mask = 3 -> nhị phân 011 -> lật các lá [0, 1]
mask = 4 -> nhị phân 100 -> lật các lá [2]
mask = 5 -> nhị phân 101 -> lật các lá [0, 2]
mask = 6 -> nhị phân 110 -> lật các lá [1, 2]
mask = 7 -> nhị phân 111 -> lật các lá [0, 1, 2]


### 3.5. Mảng đánh dấu theo giá trị (counting array)

Giá trị chỉ từ `1` đến `2000`, nên ta có thể dùng mảng `bool` độ dài `2001` thay cho set: `bad[x] = True` nghĩa là `x` bị cấm. Truy cập `O(1)`, không băm.

In [6]:
bad = [False] * 2001
for a, b in zip(fronts, backs):
    if a == b:
        bad[a] = True
print([x for x in range(2001) if bad[x]])

[1, 4]


## 4. Ý tưởng nền tảng / trực giác

Nhìn một số `x` cụ thể. Mỗi lá bài chứa `x` có thể rơi vào ba loại:

1. **Lá có `x` ở đúng một mặt** (mặt kia khác `x`): ta **chọn hướng** để `x` nằm dưới.
2. **Lá có `x` ở cả hai mặt** (`fronts[i] == backs[i] == x`): `x` **luôn** nằm ở mặt trên (dù lật hay không) → `x` **không bao giờ** là số tốt.
3. **Lá không chứa `x`**: không quan tâm.

Vậy `x` có thể là số tốt ⇔ (1) `x` xuất hiện trên **ít nhất một** lá nào đó (để có chỗ nằm dưới) và (2) **không** có lá loại 2. Khi điều kiện đủ, ta chỉ việc **lật mọi lá loại 1 sao cho `x` nằm dưới**; các lá còn lại không ảnh hưởng.

Tính tay ví dụ 1: các lá là `(1,1), (2,3), (4,4), (4,1), (7,3)`.

| lá | hai mặt giống nhau? | ý nghĩa |
|---|---|---|
| `(1,1)` | có | cấm số `1` |
| `(2,3)` | không | - |
| `(4,4)` | có | cấm số `4` |
| `(4,1)` | không | - |
| `(7,3)` | không | - |

Số bị cấm `{1, 4}`; số xuất hiện trên bài `{1, 2, 3, 4, 7}`; số có thể tốt `{2, 3, 7}`; nhỏ nhất là **`2`** ✓.

In [7]:
def explain(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    seen = set(fronts) | set(backs)
    good = sorted(seen - banned)
    print("số bị cấm       :", sorted(banned))
    print("số xuất hiện    :", sorted(seen))
    print("số có thể tốt   :", good)
    print("số tốt nhỏ nhất :", good[0] if good else 0)


explain(*examples[0][:2])
print()
explain(*examples[1][:2])

số bị cấm       : [1, 4]
số xuất hiện    : [1, 2, 3, 4, 7]
số có thể tốt   : [2, 3, 7]
số tốt nhỏ nhất : 2

số bị cấm       : [1]
số xuất hiện    : [1]
số có thể tốt   : []
số tốt nhỏ nhất : 0


## 5. Cách 1 — Brute Force: thử mọi cách lật (`2^n`)

### Ý tưởng

Với mỗi mặt nạ `mask` (mỗi cách lật), tính hai danh sách: các số **mặt trên** `up` và các số **mặt dưới** `down`. Các số tốt là `set(down) - set(up)`. Lấy số nhỏ nhất của chúng; rồi lấy nhỏ nhất qua mọi cách lật. Nếu không có số tốt nào thì trả `0`.

Cách này **đúng chắc chắn** vì làm đúng theo định nghĩa của đề, nên ta dùng làm **oracle**. Nó chỉ chạy được khi `n` nhỏ (khoảng `n <= 14`).

In [8]:
def flip_bruteforce(fronts, backs):
    n = len(fronts)
    best = None
    for mask in range(1 << n):
        up = [backs[i] if (mask >> i) & 1 else fronts[i] for i in range(n)]
        down = [fronts[i] if (mask >> i) & 1 else backs[i] for i in range(n)]
        good = set(down) - set(up)
        if good:
            m = min(good)
            if best is None or m < best:
                best = m
    return 0 if best is None else best


for fronts, backs, expected in examples:
    print(fronts, backs, "->", flip_bruteforce(fronts, backs), "| kỳ vọng", expected)
    assert flip_bruteforce(fronts, backs) == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2 | kỳ vọng 2
[1] [1] -> 0 | kỳ vọng 0


### Độ phức tạp và nút thắt

- `2^n` cách lật × `O(n)` để tính hai danh sách và tập: `O(2^n · n)` thời gian, `O(n)` bộ nhớ. Với `n = 1000` là **không thể** (`2^1000`).

Nút thắt: ta liệt kê **mọi cấu hình** trong khi chỉ quan tâm **một số `x`** tại một thời điểm. Mỗi số `x` chỉ "đòi hỏi" các lá chứa `x` được lật đúng hướng — các lá khác tuỳ ý. Vậy với mỗi `x` ta chỉ cần **một** cấu hình (cách 2), hoặc đơn giản hơn — chỉ cần biết `x` có bị cấm hay không (cách 3 trở đi).

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát.** Số `x` là số tốt có thể đạt được ⇔ `x` xuất hiện trên bài và **không** có lá nào `fronts[i] == backs[i] == x`.

Chứng minh hai chiều (sẽ chặt chẽ ở mục 10):

- (⇒) Nếu có lá `(x, x)` thì `x` luôn ở mặt trên → không tốt. Nếu `x` không xuất hiện trên lá nào thì không thể nằm ở mặt dưới.
- (⇐) Với mỗi lá chứa `x` ở đúng một mặt, lật (hoặc không) để `x` hướng xuống; mặt trên của lá đó là số khác `x`. Các lá không chứa `x` tuỳ ý. Khi đó `x` ở mặt dưới của ít nhất một lá và không ở mặt trên của lá nào.

Thuật toán: gom `banned = {a : fronts[i] == backs[i] == a}`; đáp án là `min` của mọi số xuất hiện (ở bất kỳ mặt nào) **không nằm trong** `banned`; không có thì `0`. Thời gian `O(n)`.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) vét cạn `2^n` (mục 5), (2) thử từng số ứng viên và dựng đúng cách lật, (3) tập cấm bằng hash set, (4) sắp xếp rồi lấy số hợp lệ đầu tiên, (5) mảng đánh dấu theo giá trị.

### 7.1. Cách 2 — Thử từng ứng viên, dựng cách lật rồi kiểm tra theo định nghĩa

**Giải thích từng đoạn.**

1. Lấy các ứng viên: mọi số xuất hiện trên bài, sắp xếp tăng dần.
2. Với mỗi ứng viên `x`, **dựng** cách lật: lá nào có `fronts[i] == x` thì lật (để `x` nằm dưới), các lá còn lại giữ nguyên.
3. Tính mặt trên/mặt dưới của cấu hình đó và kiểm tra **đúng định nghĩa**: `x` có nằm dưới và không nằm trên không.
4. Ứng viên đầu tiên (nhỏ nhất) thoả thì trả về; không có thì `0`.

Cách này kiểm tra theo định nghĩa nên **không cần** tin vào quan sát ở mục 6; nó đóng vai cầu nối giữa vét cạn và cách hash set. Thời gian `O(V · n)` với `V` là số giá trị khác nhau.

In [9]:
def flip_candidate_sim(fronts, backs):
    n = len(fronts)
    for x in sorted(set(fronts) | set(backs)):
        up = list(fronts)
        down = list(backs)
        for i in range(n):
            if fronts[i] == x:                       # lật lá này để x nằm dưới
                up[i], down[i] = backs[i], fronts[i]
        if x in down and x not in up:
            return x
    return 0


for fronts, backs, expected in examples:
    print(fronts, backs, "->", flip_candidate_sim(fronts, backs))
    assert flip_candidate_sim(fronts, backs) == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2
[1] [1] -> 0


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        n = len(fronts)
        for x in sorted(set(fronts) | set(backs)):
            up = list(fronts)
            down = list(backs)
            for i in range(n):
                if fronts[i] == x:
                    up[i], down[i] = backs[i], fronts[i]
            if x in down and x not in up:
                return x
        return 0
```

**Độ phức tạp:** Time `O(V · n)` (`V <= 2000`, `n <= 1000` → khoảng `2 * 10^6` — vẫn nhanh), Space `O(n)`.

### 7.2. Cách 3 — Tập số bị cấm bằng hash set `O(n)` (cách nộp)

**Giải thích từng đoạn.**

1. `banned = {a for a, b in zip(fronts, backs) if a == b}` — mọi số nằm trên một lá có hai mặt giống nhau.
2. Duyệt mọi số của cả hai mảng (`chain(fronts, backs)`), giữ những số **không** thuộc `banned`.
3. `min(..., default=0)`.

In [10]:
def flip_hash(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    return min((x for x in chain(fronts, backs) if x not in banned), default=0)


for fronts, backs, expected in examples:
    print(fronts, backs, "->", flip_hash(fronts, backs))
    assert flip_hash(fronts, backs) == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2
[1] [1] -> 0


**Code nộp LeetCode (cách 3):**

```python
from itertools import chain
from typing import List

class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        banned = {a for a, b in zip(fronts, backs) if a == b}
        return min((x for x in chain(fronts, backs) if x not in banned), default=0)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho tập `banned`.

### 7.3. Cách 4 — Sắp xếp tất cả số, lấy số hợp lệ đầu tiên

**Ý tưởng.** Gộp mọi số của hai mặt, sắp xếp tăng dần; vì ta muốn **nhỏ nhất**, chỉ cần quét từ đầu và dừng ở số đầu tiên **không bị cấm**. Ưu điểm: dừng sớm; nhược: tốn sắp xếp `O(n log n)`.

In [11]:
def flip_sorted(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    for x in sorted(chain(fronts, backs)):
        if x not in banned:
            return x
    return 0


for fronts, backs, expected in examples:
    print(fronts, backs, "->", flip_sorted(fronts, backs))
    assert flip_sorted(fronts, backs) == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2
[1] [1] -> 0


**Code nộp LeetCode (cách 4):**

```python
from itertools import chain
from typing import List

class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        banned = {a for a, b in zip(fronts, backs) if a == b}
        for x in sorted(chain(fronts, backs)):
            if x not in banned:
                return x
        return 0
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### 7.4. Cách 5 — Mảng đánh dấu theo giá trị (counting array)

**Ý tưởng.** Vì `1 <= giá trị <= 2000`, dùng hai mảng `bool` độ dài `maxv + 1`: `present[x]` (có xuất hiện) và `bad[x]` (bị cấm). Quét `x` từ `1` lên và trả `x` đầu tiên có `present[x] and not bad[x]`. Không dùng băm hay sắp xếp: `O(n + maxv)`.

In [12]:
def flip_counting(fronts, backs):
    maxv = max(max(fronts), max(backs))
    present = [False] * (maxv + 1)
    bad = [False] * (maxv + 1)
    for a, b in zip(fronts, backs):
        present[a] = present[b] = True
        if a == b:
            bad[a] = True
    for x in range(1, maxv + 1):
        if present[x] and not bad[x]:
            return x
    return 0


for fronts, backs, expected in examples:
    print(fronts, backs, "->", flip_counting(fronts, backs))
    assert flip_counting(fronts, backs) == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2
[1] [1] -> 0


**Code nộp LeetCode (cách 5):**

```python
from typing import List

class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        maxv = max(max(fronts), max(backs))
        present = [False] * (maxv + 1)
        bad = [False] * (maxv + 1)
        for a, b in zip(fronts, backs):
            present[a] = present[b] = True
            if a == b:
                bad[a] = True
        for x in range(1, maxv + 1):
            if present[x] and not bad[x]:
                return x
        return 0
```

**Độ phức tạp:** Time `O(n + maxv)`, Space `O(maxv)`.

Năm cách cho cùng đáp án; mục 12 đối chiếu chúng với oracle vét cạn trên rất nhiều input.

## 8. TRACE từng bước

Trace cách 3 (hash set) trên ví dụ 1 và ví dụ 2. Hàm `trace_flip` in từng lá bài, tập `banned` được cập nhật ra sao, rồi duyệt các ứng viên từ nhỏ đến lớn.

Bảng tay cho ví dụ 1 (`fronts = [1,2,4,4,7]`, `backs = [1,3,4,1,3]`):

| lá `i` | `(a, b)` | `a == b`? | `banned` sau lá này |
|---|---|---|---|
| 0 | `(1, 1)` | có | `{1}` |
| 1 | `(2, 3)` | không | `{1}` |
| 2 | `(4, 4)` | có | `{1, 4}` |
| 3 | `(4, 1)` | không | `{1, 4}` |
| 4 | `(7, 3)` | không | `{1, 4}` |

Ứng viên tăng dần `1, 2, 3, 4, 7`: `1` cấm, **`2` hợp lệ → đáp án `2`**.

In [13]:
def trace_flip(fronts, backs):
    banned = set()
    for i, (a, b) in enumerate(zip(fronts, backs)):
        if a == b:
            banned.add(a)
        print(f"lá {i}: ({a},{b}) {'GIỐNG -> cấm ' + str(a) if a == b else 'khác nhau'} | banned = {sorted(banned)}")
    candidates = sorted(set(fronts) | set(backs))
    print("ứng viên tăng dần:", candidates)
    for x in candidates:
        if x in banned:
            print(f"  x = {x}: bị cấm -> bỏ")
        else:
            print(f"  x = {x}: hợp lệ -> ĐÁP ÁN")
            return x
    print("  không có ứng viên hợp lệ -> 0")
    return 0


assert trace_flip([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == 2
print()
assert trace_flip([1], [1]) == 0

lá 0: (1,1) GIỐNG -> cấm 1 | banned = [1]
lá 1: (2,3) khác nhau | banned = [1]
lá 2: (4,4) GIỐNG -> cấm 4 | banned = [1, 4]
lá 3: (4,1) khác nhau | banned = [1, 4]
lá 4: (7,3) khác nhau | banned = [1, 4]
ứng viên tăng dần: [1, 2, 3, 4, 7]
  x = 1: bị cấm -> bỏ
  x = 2: hợp lệ -> ĐÁP ÁN

lá 0: (1,1) GIỐNG -> cấm 1 | banned = [1]
ứng viên tăng dần: [1]
  x = 1: bị cấm -> bỏ
  không có ứng viên hợp lệ -> 0


Để thấy **cách lật cụ thể** cho đáp án `2` ở ví dụ 1, ta in cấu hình mà cách 2 dựng ra: chỉ lá chứa `2` ở mặt trước (lá `1`) cần lật.

In [14]:
def show_flip_config(fronts, backs, x):
    up, down = list(fronts), list(backs)
    flipped = []
    for i in range(len(fronts)):
        if fronts[i] == x:
            up[i], down[i] = backs[i], fronts[i]
            flipped.append(i)
    print(f"x = {x}: lật các lá {flipped} -> mặt trên {up}, mặt dưới {down}")
    print(f"  {x} ở mặt dưới? {x in down} | {x} ở mặt trên? {x in up} | tốt? {x in down and x not in up}")
    return x in down and x not in up


assert show_flip_config([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2)
assert not show_flip_config([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 4)       # 4 bị cấm bởi lá (4,4)

x = 2: lật các lá [1] -> mặt trên [1, 3, 4, 4, 7], mặt dưới [1, 2, 4, 1, 3]
  2 ở mặt dưới? True | 2 ở mặt trên? False | tốt? True
x = 4: lật các lá [2, 3] -> mặt trên [1, 2, 4, 1, 7], mặt dưới [1, 3, 4, 4, 3]
  4 ở mặt dưới? True | 4 ở mặt trên? True | tốt? False


## 9. Code nộp LeetCode

Đây là cách hash set (cách 3) — ngắn gọn, `O(n)`.

```python
from itertools import chain
from typing import List

class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        banned = {a for a, b in zip(fronts, backs) if a == b}
        return min((x for x in chain(fronts, backs) if x not in banned), default=0)
```

In [15]:
class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        banned = {a for a, b in zip(fronts, backs) if a == b}
        return min((x for x in chain(fronts, backs) if x not in banned), default=0)


sol = Solution()
for fronts, backs, expected in examples:
    got = sol.flipgame(fronts, backs)
    print(fronts, backs, "->", got, "| kỳ vọng", expected)
    assert got == expected

[1, 2, 4, 4, 7] [1, 3, 4, 1, 3] -> 2 | kỳ vọng 2
[1] [1] -> 0 | kỳ vọng 0


## 10. Vì sao đúng

**Bổ đề.** Một số `x` có thể trở thành số tốt (với một cách lật nào đó) ⇔ `x` xuất hiện trên ít nhất một lá **và** không có lá nào `fronts[i] == backs[i] == x`.

**Chứng minh.**

(⇒) Giả sử sau một cách lật `x` tốt. Khi đó `x` nằm ở mặt dưới của một lá nên `x` xuất hiện trên bài. Nếu tồn tại lá `(x, x)` thì lá đó có `x` ở mặt trên dù lật thế nào → `x` nằm ở mặt trên của một lá, mâu thuẫn với "không nằm ở mặt trên của lá nào". Vậy không có lá `(x, x)`.

(⇐) Giả sử `x` xuất hiện trên một lá và không có lá `(x, x)`. Xét mọi lá chứa `x`: mỗi lá như vậy có `x` ở **đúng một** mặt (vì không phải `(x, x)`). Với lá có `x` ở mặt trước, ta lật; với lá có `x` ở mặt sau, ta không lật. Khi đó trong các lá chứa `x`, `x` luôn ở mặt dưới. Các lá không chứa `x` không ảnh hưởng. Vậy `x` ở mặt dưới của ít nhất một lá và không ở mặt trên của lá nào → tốt. ∎

**Hệ quả.** Tập các số tốt có thể đạt được là `(mọi số xuất hiện) − banned`, trong đó `banned = {a : fronts[i] == backs[i] == a}`. Số tốt nhỏ nhất có thể là `min` của tập này (hoặc `0` nếu rỗng) — chính là kết quả của cách 3, 4, 5. Cách 2 kiểm tra trực tiếp theo định nghĩa với cấu hình dựng ở (⇐) nên cho cùng đáp án; cách 1 liệt kê mọi cấu hình nên là chuẩn.

**Vì sao các số khác không ảnh hưởng?** Việc đặt `x` xuống chỉ ràng buộc các lá chứa `x`; ràng buộc này không "tranh chấp" với bất kỳ số nào khác vì ta chỉ cần **một** số tốt (số nhỏ nhất).

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn mọi cách lật | `O(2^n · n)` | `O(n)` | oracle, `n <= 14` |
| 2. Thử ứng viên + dựng cấu hình | `O(V · n)` | `O(n)` | kiểm tra theo định nghĩa |
| 3. Hash set cấm (`Solution`) | `O(n)` | `O(n)` | cách nộp |
| 4. Sắp xếp + số đầu tiên hợp lệ | `O(n log n)` | `O(n)` | dừng sớm |
| 5. Mảng đánh dấu theo giá trị | `O(n + maxv)` | `O(maxv)` | không băm, `maxv = 2000` |

`V` là số giá trị khác nhau (tối đa `2000`). Với `n <= 1000` và giá trị `<= 2000`, mọi cách trừ vét cạn đều chạy tức thì.

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case (1 lá, mọi lá hai mặt giống nhau, không lá nào giống nhau, số trùng nhiều lá), (c) phản ví dụ cho các lời giải sai, (d) đối chiếu **mọi cách** với oracle vét cạn `2^n` trên input nhỏ ngẫu nhiên có seed, (e) test lớn đúng ràng buộc (`n = 1000`, giá trị tới `2000`), (f) kiểm chứng đáp án bằng **cấu hình lật cụ thể**.

### 12.1. Ví dụ của đề và edge case

In [16]:
ALL = {
    "candidate_sim": flip_candidate_sim,
    "hash": flip_hash,
    "sorted": flip_sorted,
    "counting": flip_counting,
    "solution": lambda f, b: Solution().flipgame(f, b),
}

cases = [
    ([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2),       # ví dụ 1
    ([1], [1], 0),                               # ví dụ 2
    ([1], [2], 1),                               # một lá, hai mặt khác nhau: min(1,2) = 1
    ([2], [1], 1),                               # lật để mặt 1 xuống? mặt 1 dưới sẵn -> 1
    ([5, 5], [5, 5], 0),                         # mọi lá giống nhau
    ([1, 2, 3], [1, 2, 3], 0),                   # mọi lá giống nhau, nhiều số
    ([1, 1], [2, 3], 1),                         # số 1 nằm trên nhiều lá nhưng không có lá (1,1)
    ([1, 2], [2, 1], 1),                         # số xuất hiện ở cả hai mặt (khác lá) vẫn tốt được
    ([3, 3, 3], [3, 3, 4], 4),                   # 3 bị cấm; chỉ 4 còn lại
    ([2, 3, 2], [3, 3, 5], 2),                   # dùng ở bài 1
    ([1, 3, 3, 6], [3, 3, 2, 2], 1),             # dùng ở bài 2
    ([2000], [1999], 1999),                      # giá trị biên
    ([1, 2], [1, 2], 0),
    ([7, 7, 7, 7], [8, 8, 8, 8], 7),
]
for fronts, backs, expected in cases:
    assert flip_bruteforce(fronts, backs) == expected, (fronts, backs, flip_bruteforce(fronts, backs), expected)
    for name, fn in ALL.items():
        got = fn(fronts, backs)
        assert got == expected, (name, fronts, backs, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách và oracle")

tất cả 14 ca cố định đều đúng với 5 cách và oracle


### 12.2. Phản ví dụ cho các lời giải sai

Chi tiết ở mục 13; các ca làm lộ lỗi, dùng làm hồi quy:

- `([5], [3])` → đúng `3` (phải xét cả mặt sau khi tìm ứng viên).
- `([1,2,4,4,7], [1,3,4,1,3])` → đúng `2` (không loại số bị cấm sẽ ra `1`).
- `([1,1], [2,3])` → đúng `1` (số xuất hiện trên nhiều lá không bị cấm).
- `([1,2], [2,1])` → đúng `1` (số xuất hiện ở cả hai mặt nhưng trên **lá khác nhau** không bị cấm).

In [17]:
regression = [([5], [3], 3), ([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2), ([1, 1], [2, 3], 1), ([1, 2], [2, 1], 1)]
for fronts, backs, expected in regression:
    assert flip_bruteforce(fronts, backs) == expected
    for name, fn in ALL.items():
        assert fn(fronts, backs) == expected, (name, fronts, backs)
print("ca hồi quy đều đúng")

ca hồi quy đều đúng


### 12.3. Đối chiếu ngẫu nhiên nhỏ với oracle vét cạn

Giá trị nhỏ (`1..5`) để các lá **hay trùng số** và hay có lá hai mặt giống nhau; `n` từ `1` đến `9` (`2^9 = 512` cách lật).

In [18]:
rng = random.Random(822)
zero_count = 0
total = 0
for _ in range(2500):
    n = rng.randint(1, 9)
    fronts = [rng.randint(1, 5) for _ in range(n)]
    backs = [rng.randint(1, 5) for _ in range(n)]
    expected = flip_bruteforce(fronts, backs)
    zero_count += expected == 0
    total += 1
    for name, fn in ALL.items():
        assert fn(fronts, backs) == expected, (name, fronts, backs, fn(fronts, backs), expected)
print("2500 input nhỏ khớp oracle | số ca đáp án 0:", zero_count, "| số ca khác 0:", total - zero_count)
assert 0 < zero_count < total

2500 input nhỏ khớp oracle | số ca đáp án 0: 78 | số ca khác 0: 2422


### 12.4. Ngẫu nhiên với dải giá trị rộng hơn (oracle vét cạn vẫn chạy được)

`n` tới `11`, giá trị `1..20`; cũng thử ca cố ý có nhiều lá hai mặt giống nhau.

In [19]:
rng = random.Random(1822)
for _ in range(600):
    n = rng.randint(1, 11)
    hi = rng.choice([3, 8, 20])
    fronts = [rng.randint(1, hi) for _ in range(n)]
    backs = [rng.randint(1, hi) if rng.random() < 0.6 else fronts[i] for i in range(n)]   # nhiều lá (a, a)
    expected = flip_bruteforce(fronts, backs)
    for name, fn in ALL.items():
        assert fn(fronts, backs) == expected, (name, fronts, backs)
print("600 input (nhiều lá hai mặt giống nhau) khớp oracle")

600 input (nhiều lá hai mặt giống nhau) khớp oracle


### 12.5. Test lớn (`n = 1000`, giá trị tới `2000`) và kiểm chứng bằng cấu hình lật

Với `n = 1000` không có oracle vét cạn; ta (1) đối chiếu 5 cách với nhau, và (2) kiểm chứng đáp án `x > 0` bằng **cách lật dựng cụ thể** theo định nghĩa của đề; đồng thời kiểm tra không có số **nhỏ hơn** nào hợp lệ bằng cách thử riêng từng số nhỏ hơn.

In [20]:
def is_good_after_config(fronts, backs, x):
    """Dựng cấu hình đặt x xuống (lật mọi lá có x ở mặt trước) rồi kiểm tra theo định nghĩa."""
    up, down = list(fronts), list(backs)
    for i in range(len(fronts)):
        if fronts[i] == x:
            up[i], down[i] = backs[i], fronts[i]
    return x in down and x not in up


import time
t0 = time.perf_counter()
rng = random.Random(2000)
for trial in range(30):
    n = 1000
    hi = rng.choice([2, 10, 100, 2000])
    fronts = [rng.randint(1, hi) for _ in range(n)]
    backs = [rng.randint(1, hi) for _ in range(n)]
    res = {name: fn(fronts, backs) for name, fn in ALL.items()}
    assert len(set(res.values())) == 1, (hi, res)
    ans = res["solution"]
    present = sorted(set(fronts) | set(backs))                   # số không xuất hiện thì chắc chắn không tốt
    if ans:
        assert is_good_after_config(fronts, backs, ans)
        for smaller in present:
            if smaller < ans:
                assert not is_good_after_config(fronts, backs, smaller), smaller
    else:
        assert all(not is_good_after_config(fronts, backs, x) for x in present)
print("30 test n = 1000: các cách khớp nhau và đáp án khớp cấu hình lật | thời gian:", round(time.perf_counter() - t0, 2), "s")

# ca đặc biệt
n = 1000
assert Solution().flipgame([2000] * n, [2000] * n) == 0
assert Solution().flipgame([1] * n, [2000] * n) == 1
assert Solution().flipgame(list(range(1, n + 1)), list(range(1, n + 1))) == 0
assert Solution().flipgame(list(range(1, n + 1)), list(range(2, n + 2))) == 1
print("ca đặc biệt đều đúng")

30 test n = 1000: các cách khớp nhau và đáp án khớp cấu hình lật | thời gian: 0.04 s
ca đặc biệt đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lấy ứng viên từ mặt trước

In [21]:
def wrong_fronts_only(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    return min((x for x in fronts if x not in banned), default=0)    # SAI: bỏ qua mặt sau


print("sai :", wrong_fronts_only([5], [3]), "| đúng:", flip_hash([5], [3]))
assert wrong_fronts_only([5], [3]) == 5 and flip_hash([5], [3]) == 3

sai : 5 | đúng: 3


**Vì sao sai:** số ở mặt sau cũng có thể là số tốt (không lật thì nó nằm dưới). **Cách đúng:** xét cả `fronts` lẫn `backs` (`chain(fronts, backs)`).

### Lỗi 2 — Không loại số bị cấm khỏi ứng viên

In [22]:
def wrong_no_exclude(fronts, backs):
    return min(chain(fronts, backs))                                   # SAI: quên loại banned


print("sai :", wrong_no_exclude([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]), "| đúng:", flip_hash([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]))
assert wrong_no_exclude([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == 1

sai : 1 | đúng: 2


**Vì sao sai:** số `1` nằm trên lá `(1, 1)` nên luôn ở mặt trên, không thể tốt. **Cách đúng:** loại `banned` trước khi lấy `min`.

### Lỗi 3 — Cấm mọi số xuất hiện trên **nhiều lá** (đếm tần suất)

In [23]:
from collections import Counter

def wrong_frequency_ban(fronts, backs):
    cnt = Counter(chain(fronts, backs))
    banned = {x for x, c in cnt.items() if c > 1}                      # SAI: tần suất không phải điều kiện
    return min((x for x in cnt if x not in banned), default=0)


print("sai :", wrong_frequency_ban([1, 1], [2, 3]), "| đúng:", flip_hash([1, 1], [2, 3]))
assert wrong_frequency_ban([1, 1], [2, 3]) == 2 and flip_hash([1, 1], [2, 3]) == 1

sai : 2 | đúng: 1


**Vì sao sai:** số `1` nằm trên hai lá `(1,2)` và `(1,3)`; ta lật cả hai để `1` xuống dưới, mặt trên là `2` và `3` — `1` vẫn tốt. Chỉ khi **cùng một lá** có `x` ở cả hai mặt thì `x` mới bị cấm.

### Lỗi 4 — Cấm số xuất hiện ở cả hai tập mặt trước và mặt sau (nhưng trên lá khác nhau)

In [24]:
def wrong_intersection(fronts, backs):
    banned = set(fronts) & set(backs)                                  # SAI: giao hai tập, không phải cùng lá
    return min((x for x in chain(fronts, backs) if x not in banned), default=0)


print("sai :", wrong_intersection([1, 2], [2, 1]), "| đúng:", flip_hash([1, 2], [2, 1]))
assert wrong_intersection([1, 2], [2, 1]) == 0 and flip_hash([1, 2], [2, 1]) == 1

sai : 0 | đúng: 1


**Vì sao sai:** ở `([1,2], [2,1])`, `1` có ở mặt trước lá 0 và mặt sau lá 1 — **hai lá khác nhau**. Lật lá 0 thì `1` xuống dưới; lá 1 giữ nguyên thì `1` ở dưới. Mặt trên là `[2, 2]` → `1` tốt. Điều kiện cấm phải cùng một chỉ số `i`: `fronts[i] == backs[i]`.

### Lỗi 5 — `min` của dãy rỗng

In [25]:
def wrong_min_empty(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    return min(x for x in chain(fronts, backs) if x not in banned)     # SAI: thiếu default=0


try:
    wrong_min_empty([1], [1])
except ValueError as e:
    print("ValueError:", e)
print("đúng:", flip_hash([1], [1]))

ValueError: min() iterable argument is empty
đúng: 0


**Cách đúng:** `min(..., default=0)` (hoặc kiểm tra rỗng rồi trả `0`) — đề yêu cầu trả `0` khi không có số tốt.

### Lỗi 6 (bonus) — Độ ưu tiên toán tử tập hợp: `-` mạnh hơn `|`

Trong Python, phép hiệu `-` có độ ưu tiên **cao hơn** phép hợp `|`. Do đó `set(a) | set(b) - banned` nghĩa là `set(a) | (set(b) - banned)`, **không** phải `(set(a) | set(b)) - banned`.

In [26]:
fronts, backs = [1, 2, 4, 4, 7], [1, 3, 4, 1, 3]
banned = {a for a, b in zip(fronts, backs) if a == b}
wrong = set(fronts) | set(backs) - banned          # = set(fronts) | (set(backs) - banned)
right = (set(fronts) | set(backs)) - banned
print("sai :", sorted(wrong), "-> min =", min(wrong))
print("đúng:", sorted(right), "-> min =", min(right))
assert min(wrong) == 1 and min(right) == 2

sai : [1, 2, 3, 4, 7] -> min = 1
đúng: [2, 3, 7] -> min = 2


**Cách đúng:** luôn thêm ngoặc `(A | B) - C` khi trộn hai phép toán tập hợp.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "tưởng vét cạn nhưng thật ra là lọc":**

- Mỗi phần tử có hai lựa chọn độc lập (`2^n` cấu hình) nhưng câu hỏi chỉ quan tâm **một giá trị `x`**: hãy hỏi "điều kiện nào làm `x` **không thể** đạt?".
- Câu trả lời thường là một **điều kiện cục bộ** (ở đây: một lá có hai mặt cùng bằng `x`) → gom thành tập "cấm" rồi lọc.

**Mẹo nhớ:**

1. Quan sát then chốt: `x` bị cấm ⇔ có lá `(x, x)`.
2. Số đạt được = (mọi số xuất hiện) − (số bị cấm).
3. Dùng `min(..., default=0)` thay cho kiểm tra rỗng.
4. Luôn viết oracle vét cạn `2^n` cho input nhỏ để kiểm tra quan sát của mình.
5. Cẩn thận độ ưu tiên `-` và `|` của set; thêm ngoặc.
6. Họ hàng: LeetCode 2441, 1010, 1217, 948 (Bag of Tokens), các bài "xét điều kiện bị cấm rồi lọc".

## 15. Biến thể

### 15.1. Tập **mọi** số tốt có thể đạt được và số tốt **lớn nhất**

Từ mục 10: tập số đạt được là `(mọi số xuất hiện) − banned`. Đối chiếu với tập thu được bằng cách gộp số tốt của mọi cấu hình vét cạn.

In [27]:
def all_good_numbers(fronts, backs):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    return sorted((set(fronts) | set(backs)) - banned)


def all_good_bruteforce(fronts, backs):
    n = len(fronts)
    result = set()
    for mask in range(1 << n):
        up = [backs[i] if (mask >> i) & 1 else fronts[i] for i in range(n)]
        down = [fronts[i] if (mask >> i) & 1 else backs[i] for i in range(n)]
        result |= set(down) - set(up)
    return sorted(result)


def max_good_number(fronts, backs):
    return max(all_good_numbers(fronts, backs), default=0)


assert all_good_numbers([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == [2, 3, 7]
assert max_good_number([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == 7
rng = random.Random(151)
for _ in range(500):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 6) for _ in range(n)]
    b = [rng.randint(1, 6) for _ in range(n)]
    assert all_good_numbers(f, b) == all_good_bruteforce(f, b), (f, b)
print("15.1 OK:", all_good_numbers([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]))

15.1 OK: [2, 3, 7]


### 15.2. **Số lá tối thiểu phải lật** để `x` trở thành số tốt

Với `x` hợp lệ, các lá có `x` ở mặt **trước** buộc phải lật, các lá có `x` ở mặt sau không cần. Vậy số lật tối thiểu bằng số lá có `fronts[i] == x`. Nếu `x` bị cấm hay không xuất hiện thì không thể (`-1`). Đối chiếu bằng vét cạn (số bit `1` nhỏ nhất của mặt nạ làm `x` tốt).

In [28]:
def min_flips_for(fronts, backs, x):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    if x in banned or x not in set(fronts) | set(backs):
        return -1
    return sum(1 for a in fronts if a == x)


def min_flips_bruteforce(fronts, backs, x):
    n = len(fronts)
    best = None
    for mask in range(1 << n):
        up = [backs[i] if (mask >> i) & 1 else fronts[i] for i in range(n)]
        down = [fronts[i] if (mask >> i) & 1 else backs[i] for i in range(n)]
        if x in down and x not in up:
            c = bin(mask).count("1")
            best = c if best is None else min(best, c)
    return -1 if best is None else best


assert min_flips_for([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2) == 1
assert min_flips_for([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 3) == 0
assert min_flips_for([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 4) == -1
rng = random.Random(152)
for _ in range(500):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 5) for _ in range(n)]
    b = [rng.randint(1, 5) for _ in range(n)]
    for x in range(1, 7):
        assert min_flips_for(f, b, x) == min_flips_bruteforce(f, b, x), (f, b, x)
print("15.2 OK")

15.2 OK


### 15.3. Đếm số **cách lật** làm `x` trở thành số tốt

Các lá chứa `x` (đều có `x` ở đúng một mặt) bị ép về **một** hướng; các lá không chứa `x` tuỳ ý. Nếu có `c` lá chứa `x` thì số cách lật là `2^(n - c)`.

In [29]:
def count_flip_ways(fronts, backs, x):
    banned = {a for a, b in zip(fronts, backs) if a == b}
    if x in banned:
        return 0
    c = sum(1 for a, b in zip(fronts, backs) if a == x or b == x)
    return 0 if c == 0 else 2 ** (len(fronts) - c)


def count_ways_bruteforce(fronts, backs, x):
    n = len(fronts)
    total = 0
    for mask in range(1 << n):
        up = [backs[i] if (mask >> i) & 1 else fronts[i] for i in range(n)]
        down = [fronts[i] if (mask >> i) & 1 else backs[i] for i in range(n)]
        total += (x in down and x not in up)
    return total


assert count_flip_ways([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2) == 2 ** 4
rng = random.Random(153)
for _ in range(400):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 5) for _ in range(n)]
    b = [rng.randint(1, 5) for _ in range(n)]
    for x in range(1, 7):
        assert count_flip_ways(f, b, x) == count_ways_bruteforce(f, b, x), (f, b, x)
print("15.3 OK")

15.3 OK


### 15.4. Số tốt nhỏ thứ `k` đạt được

Sắp xếp danh sách ở 15.1 và lấy phần tử thứ `k` (đếm từ `1`); không đủ thì `0`.

In [30]:
def kth_good_number(fronts, backs, k):
    goods = all_good_numbers(fronts, backs)
    return goods[k - 1] if 1 <= k <= len(goods) else 0


assert [kth_good_number([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], k) for k in range(1, 5)] == [2, 3, 7, 0]
print([kth_good_number([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], k) for k in range(1, 5)])

[2, 3, 7, 0]


### 15.5. Biến thể "ba mặt" — mỗi lá có ba số, chỉ một số hướng lên

Tổng quát hoá: lá `i` có `m` số, ta chọn **một** số cho hướng lên, các số còn lại ở dưới. `x` là tốt nếu nằm dưới ở ít nhất một lá và không hướng lên ở lá nào. Một lá chỉ **cấm** `x` khi **mọi** số của lá đều bằng `x` (lá không còn chọn nào khác để đưa `x` xuống). Kiểm chứng bằng vét cạn các lựa chọn.

In [31]:
from itertools import product


def multi_face_min_good(cards):
    banned = {c[0] for c in cards if len(set(c)) == 1}
    values = {v for c in cards for v in c}
    return min((v for v in values if v not in banned), default=0)


def multi_face_bruteforce(cards):
    best = None
    for choice in product(*[range(len(c)) for c in cards]):          # chọn mặt hướng lên cho mỗi lá
        up = {cards[i][j] for i, j in enumerate(choice)}
        down = {v for i, c in enumerate(cards) for jj, v in enumerate(c) if jj != choice[i]}
        good = down - up
        if good:
            best = min(good) if best is None else min(best, min(good))
    return 0 if best is None else best


assert multi_face_min_good([(1, 2, 3), (2, 2, 2)]) == 1
rng = random.Random(155)
for _ in range(400):
    cards = [tuple(rng.randint(1, 4) for _ in range(3)) for _ in range(rng.randint(1, 4))]
    assert multi_face_min_good(cards) == multi_face_bruteforce(cards), cards
print("15.5 OK: tổng quát hoá cho lá có 3 mặt vẫn là 'lá mà mọi mặt bằng x thì cấm x'")

15.5 OK: tổng quát hoá cho lá có 3 mặt vẫn là 'lá mà mọi mặt bằng x thì cấm x'


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `2^n` | thử mọi cách lật | `O(2^n · n)` | `O(n)` | oracle, `n <= 14` |
| 2. Thử ứng viên + dựng cấu hình | kiểm tra đúng định nghĩa cho từng `x` | `O(V · n)` | `O(n)` | khi chưa tin vào quan sát |
| 3. Hash set cấm | `x` bị cấm ⇔ lá `(x, x)` | `O(n)` | `O(n)` | **cách nộp** |
| 4. Sắp xếp | số đầu tiên không bị cấm | `O(n log n)` | `O(n)` | khi muốn dừng sớm |
| 5. Mảng đánh dấu | `present[x]`, `bad[x]` | `O(n + maxv)` | `O(maxv)` | giá trị nhỏ, tránh băm |

Khuyến nghị: nộp cách 3 (hoặc 5); dùng cách 1 làm oracle khi tự thay đổi đề.

In [32]:
rng = random.Random(16)
for _ in range(500):
    n = rng.randint(1, 10)
    f = [rng.randint(1, 7) for _ in range(n)]
    b = [rng.randint(1, 7) for _ in range(n)]
    res = {name: fn(f, b) for name, fn in ALL.items()}
    res["bruteforce"] = flip_bruteforce(f, b)
    assert len(set(res.values())) == 1, (f, b, res)
print("500 input: 6 hàm cho cùng đáp án")

500 input: 6 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(sorted({a for a, b in zip([2, 3, 2], [3, 3, 5]) if a == b}))
print(flip_hash([2, 3, 2], [3, 3, 5]))
print(flip_hash([5, 5], [5, 5]))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `fronts = [1, 3, 3, 6]` và `backs = [3, 3, 2, 2]`: (a) tập số bị cấm; (b) tập số có thể là số tốt; (c) đáp án; (d) số lá phải lật để đạt đáp án.

In [34]:
# Bài 2: điền kết quả tính tay
hand_banned = None        # list đã sắp xếp
hand_good = None          # list đã sắp xếp
hand_answer = None
hand_flips = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách hash set.

In [35]:
def exercise_3_flipgame(fronts: List[int], backs: List[int]) -> int:
    # TODO 1: banned = tập các a sao cho a == b trên cùng một lá (dùng zip)
    # TODO 2: duyệt mọi số của cả fronts và backs, giữ số KHÔNG thuộc banned
    # TODO 3: trả về số nhỏ nhất, hoặc 0 nếu không có số nào (dùng min(..., default=0))
    return None


# assert exercise_3_flipgame([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == 2
# assert exercise_3_flipgame([1], [1]) == 0
# assert exercise_3_flipgame([1, 2], [2, 1]) == 1

### Bài 4 — Tự viết lại (vừa-khó): số lá lật tối thiểu cho một số `x`

Không nhìn lại mục 15.2, hãy tự viết `exercise_4_min_flips(fronts, backs, x)`: trả về số lá **ít nhất** phải lật để `x` trở thành số tốt; trả `-1` nếu không thể.

```python
exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2) -> 1
exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 3) -> 0
exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 4) -> -1
exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 9) -> -1
```

In [36]:
def exercise_4_min_flips(fronts: List[int], backs: List[int], x: int) -> int:
    # TODO
    pass


# assert exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2) == 1
# assert exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 3) == 0
# assert exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 4) == -1
# assert exercise_4_min_flips([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 9) == -1

### Bài 5 — Mini challenge: tối đa `k` lần lật (khó)

Viết `exercise_5_at_most_k(fronts, backs, k)` trả về **số tốt nhỏ nhất** có thể có khi bạn được lật **tối đa `k` lá**; không có thì `0`.

```python
exercise_5_at_most_k([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 0) -> 3
exercise_5_at_most_k([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 1) -> 2
exercise_5_at_most_k([5], [3], 0) -> 3
exercise_5_at_most_k([5], [5], 3) -> 0
```

Gợi ý hướng nghĩ: kết hợp bài 4 — duyệt `x` tăng dần, `x` hợp lệ nếu số lật tối thiểu của nó `<= k`.

In [37]:
def exercise_5_at_most_k(fronts: List[int], backs: List[int], k: int) -> int:
    # TODO
    pass


# assert exercise_5_at_most_k([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 0) == 3
# assert exercise_5_at_most_k([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 1) == 2
# assert exercise_5_at_most_k([5], [3], 0) == 3
# assert exercise_5_at_most_k([5], [5], 3) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: lá `(a, b)` có `a == b` sinh ra số bị cấm.
- Mức 2: các lá là `(2,3), (3,3), (2,5)`; chỉ lá thứ hai có hai mặt giống nhau. Ở dòng 3, mọi lá đều giống nhau.
- Mức 3: dòng 1 là `[3]`; dòng 2 là `2`; dòng 3 là `0`.

**Bài 2**
- Mức 1: liệt kê các lá `(1,3), (3,3), (3,2), (6,2)`.
- Mức 2: chỉ `(3,3)` hai mặt giống nhau → cấm `3`. Số xuất hiện: `1, 2, 3, 6`.
- Mức 3: (a) `[3]`; (b) `[1, 2, 6]`; (c) `1`; (d) `1` (lá `(1,3)` có `1` ở mặt trước nên phải lật).

**Bài 3**
- Mức 1: giống mục 7.2.
- Mức 2: `banned = {a for a, b in zip(fronts, backs) if a == b}`.
- Mức 3: `return min((x for x in chain(fronts, backs) if x not in banned), default=0)`.

**Bài 4**
- Mức 1: trước hết kiểm tra `x` bị cấm hoặc không xuất hiện → `-1`.
- Mức 2: lá có `x` ở mặt trước phải lật; lá có `x` ở mặt sau thì không.
- Mức 3: kết quả `sum(1 for a in fronts if a == x)`.

**Bài 5**
- Mức 1: duyệt các ứng viên từ nhỏ đến lớn.
- Mức 2: với mỗi `x`, dùng `exercise_4_min_flips`; nếu `!= -1` và `<= k` thì trả `x`.
- Mức 3: duyệt `sorted(set(fronts) | set(backs))` (có ngoặc!); không tìm được thì trả `0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(sorted({a for a, b in zip([2, 3, 2], [3, 3, 5]) if a == b}))
print(flip_hash([2, 3, 2], [3, 3, 5]))
print(flip_hash([5, 5], [5, 5]))

assert sorted({a for a, b in zip([2, 3, 2], [3, 3, 5]) if a == b}) == [3]
assert flip_hash([2, 3, 2], [3, 3, 5]) == 2
assert flip_hash([5, 5], [5, 5]) == 0

[3]
2
0


### Đáp án Bài 2

Các lá: `(1,3), (3,3), (3,2), (6,2)`.

| lá | giống nhau? |
|---|---|
| `(1,3)` | không |
| `(3,3)` | **có** → cấm `3` |
| `(3,2)` | không |
| `(6,2)` | không |

Số bị cấm `[3]`; số có thể tốt `[1, 2, 6]`; đáp án `1`; số lá phải lật `1` (lá `(1,3)`).

In [39]:
f2, b2 = [1, 3, 3, 6], [3, 3, 2, 2]
trace_flip(f2, b2)
assert sorted({a for a, b in zip(f2, b2) if a == b}) == [3]
assert all_good_numbers(f2, b2) == [1, 2, 6]
assert flip_hash(f2, b2) == 1
assert min_flips_for(f2, b2, 1) == 1 == min_flips_bruteforce(f2, b2, 1)
print("Bài 2 OK")

lá 0: (1,3) khác nhau | banned = []
lá 1: (3,3) GIỐNG -> cấm 3 | banned = [3]
lá 2: (3,2) khác nhau | banned = [3]
lá 3: (6,2) khác nhau | banned = [3]
ứng viên tăng dần: [1, 2, 3, 6]
  x = 1: hợp lệ -> ĐÁP ÁN
Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_flipgame_solution(fronts: List[int], backs: List[int]) -> int:
    banned = {a for a, b in zip(fronts, backs) if a == b}
    return min((x for x in chain(fronts, backs) if x not in banned), default=0)


assert exercise_3_flipgame_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3]) == 2
assert exercise_3_flipgame_solution([1], [1]) == 0
assert exercise_3_flipgame_solution([1, 2], [2, 1]) == 1
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 5) for _ in range(n)]
    b = [rng.randint(1, 5) for _ in range(n)]
    assert exercise_3_flipgame_solution(f, b) == flip_bruteforce(f, b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_min_flips_solution(fronts: List[int], backs: List[int], x: int) -> int:
    banned = {a for a, b in zip(fronts, backs) if a == b}
    if x in banned or x not in (set(fronts) | set(backs)):
        return -1
    return sum(1 for a in fronts if a == x)


assert exercise_4_min_flips_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 2) == 1
assert exercise_4_min_flips_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 3) == 0
assert exercise_4_min_flips_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 4) == -1
assert exercise_4_min_flips_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 9) == -1
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 5) for _ in range(n)]
    b = [rng.randint(1, 5) for _ in range(n)]
    for x in range(1, 7):
        assert exercise_4_min_flips_solution(f, b, x) == min_flips_bruteforce(f, b, x)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_at_most_k_solution(fronts: List[int], backs: List[int], k: int) -> int:
    for x in sorted(set(fronts) | set(backs)):
        need = exercise_4_min_flips_solution(fronts, backs, x)
        if need != -1 and need <= k:
            return x
    return 0


def at_most_k_bruteforce(fronts, backs, k):
    n = len(fronts)
    best = None
    for mask in range(1 << n):
        if bin(mask).count("1") > k:
            continue
        up = [backs[i] if (mask >> i) & 1 else fronts[i] for i in range(n)]
        down = [fronts[i] if (mask >> i) & 1 else backs[i] for i in range(n)]
        good = set(down) - set(up)
        if good:
            best = min(good) if best is None else min(best, min(good))
    return 0 if best is None else best


assert exercise_5_at_most_k_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 0) == 3
assert exercise_5_at_most_k_solution([1, 2, 4, 4, 7], [1, 3, 4, 1, 3], 1) == 2
assert exercise_5_at_most_k_solution([5], [3], 0) == 3
assert exercise_5_at_most_k_solution([5], [5], 3) == 0
rng = random.Random(5)
for _ in range(400):
    n = rng.randint(1, 8)
    f = [rng.randint(1, 5) for _ in range(n)]
    b = [rng.randint(1, 5) for _ in range(n)]
    k = rng.randint(0, n)
    assert exercise_5_at_most_k_solution(f, b, k) == at_most_k_bruteforce(f, b, k), (f, b, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from itertools import chain
from typing import List

class Solution:
    def flipgame(self, fronts: List[int], backs: List[int]) -> int:
        banned = {a for a, b in zip(fronts, backs) if a == b}      # lá có hai mặt giống nhau -> cấm số đó
        return min((x for x in chain(fronts, backs) if x not in banned), default=0)
```

Câu thần chú: **"lá hai mặt giống nhau cấm số đó; đáp án là số nhỏ nhất (ở bất kỳ mặt nào) không bị cấm; không có thì 0."**

### Checklist trước khi submit

- [ ] Điều kiện cấm là `fronts[i] == backs[i]` trên **cùng một lá** (không phải giao hai tập, không phải đếm tần suất).
- [ ] Ứng viên lấy từ **cả hai** mảng (`chain(fronts, backs)`).
- [ ] Loại số bị cấm trước khi lấy `min`.
- [ ] Dùng `min(..., default=0)` để tránh `ValueError` khi không có số tốt.
- [ ] Có ngoặc khi trộn `|` và `-` của set.
- [ ] Test: một lá `(1,1)` (đáp án `0`), `([5],[3])` (đáp án `3`), `([1,2],[2,1])` (đáp án `1`), ví dụ 1 (đáp án `2`).
- [ ] Skeleton đúng: `class Solution`, method `flipgame(self, fronts: List[int], backs: List[int]) -> int`; độ phức tạp `O(n)`.